<a href="https://colab.research.google.com/github/hamzafarooq/multi-agent-course/blob/main/modules/Module_5_Multi_Agents/how_skills_work_under_the_hood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Under the Hood: How an Agent Skill Becomes an SDK App

### A guided tour of `market-sizing-sdk`

You've probably *used* an Agent Skill inside Claude Code or Claude.ai — you ask a question, the right skill quietly activates, and you get a well-structured answer. It can feel like magic.

This notebook pulls back the curtain. We'll take the **`market-sizing`** skill and show that it is — at its core — **just a markdown file full of instructions**. Then we'll reproduce its behavior from scratch using the raw **Anthropic Python SDK**, one small step at a time.

By the end you'll be able to answer:

1. What *is* a skill, physically? (Spoiler: a text file.)
2. What does the YAML frontmatter do, and why do we strip it?
3. How does an instruction file become a **system prompt**?
4. What does an Anthropic API call actually look like — request and response?
5. What's the difference between running a skill *in Claude Code* vs. *via the SDK*?

> **Running on Google Colab?** Just hit the badge above (or *Runtime → Run all*). The Setup section installs the SDK, downloads the skill file straight from GitHub, and reads your API key from Colab **Secrets** — no local files needed. If you're running locally instead, keep this notebook inside the `market-sizing-sdk/` folder so `skill/SKILL.md` resolves.

## 1. The big idea: a skill is just instructions

An Agent Skill is a folder with a `SKILL.md` file. That file has two parts:

```
---
name: market-sizing
description: Estimate TAM, SAM, and SOM ...   <-- YAML frontmatter (metadata)
---
                                              <-- everything below is the BODY
Ask the user: "Describe your product ..."
Then produce a market sizing ...
```

- **Frontmatter** (between the `---` fences): *metadata*. The `name` identifies the skill; the `description` is what Claude Code reads to decide **when to trigger** this skill. It is *routing* information — it is **not** instructions for how to answer.
- **Body** (everything after the second `---`): the actual *instructions*. This is the part that tells the model how to think, what steps to follow, and how to format the answer.

The whole trick of turning a skill into an SDK app is this one sentence:

> **The body of the skill becomes the `system` prompt of an API call.**

Let's prove it.

## 2. Setup

Three small things, all Colab-friendly:

1. **Install the SDK** — the same two packages in `requirements.txt`.
2. **Get the skill file** — download `SKILL.md` straight from GitHub so this works on a fresh Colab runtime with no local files.
3. **Load your API key from Colab Secrets** — never paste keys into a notebook.

### 2a. Install dependencies

In [ ]:
%pip install -q anthropic python-dotenv

### 2b. Download the skill file from GitHub

On Colab there's no local `skill/SKILL.md`, so we pull it from the repo's raw URL and save it to the same path the rest of the notebook (and `app.py`) expects: `skill/SKILL.md`.

If you're running locally and the file already exists, this just leaves it in place.

In [ ]:
import os
import urllib.request
from pathlib import Path

# Raw GitHub URL of the skill that ships with market-sizing-sdk.
RAW_URL = "https://github.com/hamzafarooq/multi-agent-course/blob/main/.claude/skills/market-sizing/SKILL.md"
skill_path = Path("skill/SKILL.md")

if not skill_path.exists():
    skill_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(RAW_URL, skill_path)
    print(f"Downloaded skill -> {skill_path}")
else:
    print(f"Skill already present -> {skill_path}")

### 2c. Load your API key

**On Colab:** open the **🔑 Secrets** panel in the left sidebar, add a secret named `ANTHROPIC_API_KEY` with your key as the value, and toggle *Notebook access* on. The cell below reads it with `userdata.get(...)`.

**Locally:** it falls back to a `.env` file / environment variable, then to a hidden prompt. Your key is never printed.

In [ ]:
from anthropic import Anthropic

api_key = None

# 1. Google Colab Secrets (the recommended path on Colab).
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass

# 2. Local fallback: .env file or an exported environment variable.
if not api_key:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except Exception:
        pass
    api_key = os.environ.get("ANTHROPIC_API_KEY")

# 3. Last resort: prompt without echoing the key to the screen.
if not api_key:
    from getpass import getpass
    api_key = getpass("Enter your ANTHROPIC_API_KEY: ")

client = Anthropic(api_key=api_key)

MODEL = "claude-sonnet-4-6"   # same model app.py uses
MAX_TOKENS = 2000

# The skill returns markdown (headings, tables, bold). Render it as rich
# markdown instead of printing raw text, so the output looks the way it
# would inside Claude Code.
from IPython.display import Markdown, display

def show(text):
    """Render a string as rich markdown in the notebook output."""
    display(Markdown(text))

print("Client ready. Using model:", MODEL)

## 3. Step one — read the skill as raw text

Nothing clever here. We open the exact same file `app.py` opens and print it verbatim. Notice the `---` fences at the top: that's the frontmatter we talked about.

In [ ]:
raw = Path("skill/SKILL.md").read_text(encoding="utf-8")
print(raw)

## 4. Step two — separate the frontmatter from the body

`app.py` does this in two lines:

```python
if raw.startswith("---"):
    raw = raw.split("---", 2)[-1]   # drop the frontmatter block
SYSTEM_PROMPT = raw.strip()
```

Why `split("---", 2)`? The `2` means *split on at most the first two `---`*, giving three pieces:

| piece | content |
|-------|---------|
| `[0]` | empty string (the file starts with `---`) |
| `[1]` | the YAML frontmatter |
| `[2]` | **the body** — what we want (`[-1]` grabs it) |

Let's look at each piece so the mechanic is obvious.

In [ ]:
parts = raw.split("---", 2)
print("Number of pieces:", len(parts))
print("\n--- piece [0] (before first fence) ---")
print(repr(parts[0]))
print("\n--- piece [1] (the frontmatter / metadata) ---")
print(parts[1])
print("\n--- piece [2] (the body / instructions) ---")
print(parts[2][:200], "...")

### What was in the frontmatter, and why we throw it away

The frontmatter held `name` and `description`. Inside Claude Code those fields are used to **decide whether to load the skill at all** — the model reads dozens of skill descriptions and picks the matching one. That's a *routing* job.

But once we've *already decided* to run this skill (which we have — we're building a dedicated app for it), the routing metadata is no longer needed. The instructions are everything below. So we drop the frontmatter and keep the body.

In [ ]:
# Reproduce exactly what app.py builds:
body = raw
if body.startswith("---"):
    body = body.split("---", 2)[-1]
SYSTEM_PROMPT = body.strip()

print("SYSTEM_PROMPT starts with:\n")
print(SYSTEM_PROMPT[:300])
print("\n...\nSanity check — does it start with '---'? ->", SYSTEM_PROMPT.startswith("---"))

## 5. Step three — the two halves of an API call: `system` vs `messages`

Every Anthropic chat call has two distinct inputs that play very different roles:

- **`system`** — standing instructions that shape *how* the model behaves for the whole conversation. This is where our skill body goes. The user never types this; it's the "operating manual."
- **`messages`** — the actual back-and-forth: a list of `{"role": "user" | "assistant", "content": ...}` turns. This is where the user's question goes.

So the mapping from "skill" to "API call" is:

| Skill concept | API field |
|---------------|-----------|
| Skill body (instructions) | `system` |
| The user's question | `messages` (a `user` turn) |
| The model's answer | comes back as an `assistant` turn |

Let's make our first call.

In [ ]:
question = "A scheduling app for independent hair stylists in the US."

response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    system=SYSTEM_PROMPT,                       # <-- the skill body
    messages=[{"role": "user", "content": question}],  # <-- the user's question
)

# Don't format it yet — let's look at the RAW object the SDK returns.
print(type(response))
print(response)

## 6. Anatomy of the response

That raw object is a `Message`. The fields worth knowing:

- `response.role` → always `"assistant"`.
- `response.stop_reason` → why generation stopped (`"end_turn"`, `"max_tokens"`, etc.).
- `response.usage` → token counts (input + output) — this is what you're billed on.
- `response.content` → a **list of content blocks**, *not* a plain string.

Why a list of blocks? Because a response can contain more than text (e.g. tool-use blocks, thinking blocks). For a plain text answer it's a single block of `type == "text"`. To get the answer as a string, we walk the list and concatenate the text blocks — exactly what `app.py`'s `ask()` does:

```python
"".join(b.text for b in msg.content if b.type == "text")
```

In [ ]:
print("role:        ", response.role)
print("stop_reason: ", response.stop_reason)
print("usage:       ", response.usage)
print("num blocks:  ", len(response.content))
print("block 0 type:", response.content[0].type)

# Pull out the text exactly like app.py does:
answer = "".join(b.text for b in response.content if b.type == "text")
print("================ THE ANSWER (rendered as markdown) ================")
show(answer)

## 7. Wrap it in a function — this *is* the SDK app

We now have every moving part. Bundling them into one helper gives us the heart of `app.py`. The only difference from the real file is that `app.py` adds the CLI plumbing (reading `sys.argv`, handling piped input, the interactive loop). The *intelligence* is just this:

In [ ]:
def ask(history):
    """Send the conversation so far to Claude and return the assistant's text."""
    msg = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=SYSTEM_PROMPT,
        messages=history,
    )
    return "".join(b.text for b in msg.content if b.type == "text")


# One-shot, just like `python app.py "a question"`:
show(ask([{"role": "user", "content": "An AI code-review assistant for mid-size software teams."}]))

## 8. Multi-turn: why we keep a `history` list

The model is **stateless** — it remembers nothing between calls. The only reason a conversation "remembers" earlier turns is that we resend the entire history every time.

To have a follow-up conversation we append each turn to a list:

1. add the user's message,
2. call `ask(history)`,
3. append the assistant's reply,
4. repeat.

This is exactly the `interactive()` loop in `app.py`. Watch the second question rely on context from the first.

In [ ]:
history = []

def turn(user_text):
    history.append({"role": "user", "content": user_text})
    reply = ask(history)
    history.append({"role": "assistant", "content": reply})
    show(f"**USER:** {user_text}")
    show(reply)
    show("---")

turn("A meal-kit subscription for busy urban professionals in India.")
turn("Now redo just the SOM assuming we only launch in Mumbai and Delhi for year one.")

In [ ]:
# Peek at the history object that powers the 'memory'.
for i, m in enumerate(history):
    preview = m["content"][:70].replace("\n", " ")
    print(f"{i}. {m['role']:9} | {preview}...")

## 9. The payoff: prove the skill is doing the work

Is the structured TAM/SAM/SOM output really coming from the *skill*, or would the model just do that anyway? Let's run an **ablation** — the same question with **no system prompt** — and compare.

This is the single best way to *see* what a skill buys you.

In [ ]:
plain = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    # system intentionally omitted
    messages=[{"role": "user", "content": "A scheduling app for independent hair stylists in the US."}],
)
plain_text = "".join(b.text for b in plain.content if b.type == "text")
print("WITHOUT the skill (no system prompt):")
show(plain_text)

Run the cell above, then scroll back to the Section 5 answer (the one *with* `system=SYSTEM_PROMPT`).

**What to look for:**

- *Without* the skill, the model often asks what you want, or gives a loose, unstructured take — it doesn't know you want TAM/SAM/SOM.
- *With* the skill, you get the exact bottom-up structure, the confidence rating, and the "use ranges, not false precision" discipline — because the skill body **told it to**.

Same model, same question. The only difference is the instruction file we loaded into `system`. **That** is what a skill is.

## 10. Claude Code vs. the SDK — what's the same, what's different

| | Inside Claude Code / Claude.ai | This SDK app |
|---|---|---|
| Skill body → behavior | ✅ used as instructions | ✅ used as the `system` prompt |
| `description` frontmatter | **Triggers** the skill automatically | Ignored — *you* chose to run it |
| Tools (file read/write, bash, scripts) | ✅ available | ❌ text in / text out only |
| Conversation memory | managed for you | you maintain the `history` list |
| Where it runs | the Claude app/CLI | anywhere Python runs (cron, server, CI) |

The crucial limitation: this wrapper is a **plain text-in / text-out** call. It faithfully reproduces *instruction-only* skills — ones that just guide reasoning and formatting, like `market-sizing`. A skill that needs to **act** (run scripts, read files, execute bash) would need **tool use / function calling** added to `app.py` — out of scope for this simple wrapper.

## 11. Your turn — experiments

Try these to deepen the intuition:

1. **Edit the skill, see the behavior change.** Open `skill/SKILL.md`, change a formatting rule (e.g. ask for a one-paragraph summary at the top), re-run Sections 4–5, and watch the output follow your edit. No code changes needed — *the instructions are the program*.
2. **Tighten the model.** Lower `MAX_TOKENS` to `300` and see how the skill copes with less room.
3. **Swap the skill.** Point `Path("skill/SKILL.md")` at a *different* skill's `SKILL.md` (e.g. `okr-writer`) and you've built a different app with zero code changes.
4. **Add a system-prompt prefix.** Try `SYSTEM_PROMPT = "Always answer in British English.\n\n" + SYSTEM_PROMPT` to feel how layered instructions stack.

### Key takeaways

- A skill is **just instructions in a markdown file**. No magic.
- **Frontmatter = routing metadata; body = the instructions.** The SDK app keeps the body, drops the frontmatter.
- The body becomes the **`system`** prompt; the user's question is a **`user`** message; the answer comes back as **content blocks** you concatenate.
- The model is **stateless** — conversation memory is just a `history` list you resend.
- A skill changes behavior *without changing code*. That's the whole point.